In [ ]:
!pip install -q openai tqdm
import json, re, time, random
from collections import defaultdict, Counter
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
MODEL = "gpt-4o"

N     = 4       # 시드 1건당 변형 수
TEMP  = 0.9

In [ ]:
# 시드 로드 + train/valid 분할
raw = json.load(open('rpt_seed_golden.json', encoding='utf-8'))
seed_all = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw

by_label = defaultdict(list)
for r in seed_all:
    by_label[r['output']['label']].append(r)

random.seed(42)
seed_train, seed_valid = [], []
VALID_RATIO = 0.30
for label, items in by_label.items():
    random.shuffle(items)
    n_valid = max(1, round(len(items) * VALID_RATIO))
    seed_valid += items[:n_valid]
    seed_train += items[n_valid:]

random.shuffle(seed_train); random.shuffle(seed_valid)
print(f"train {len(seed_train)} / valid {len(seed_valid)}")
print("train:", dict(Counter(r['output']['label'] for r in seed_train)))
print("valid:", dict(Counter(r['output']['label'] for r in seed_valid)))

json.dump(seed_valid, open('RPT_valid.json','w'), ensure_ascii=False, indent=2)
print("저장: RPT_valid.json (순수, 증강 안 함)")

In [ ]:
 # 증강 프롬프트
RPT_AUG_PROMPT = """당신은 공공 SW '사업추진결과보고서(RPT)' 검토 학습데이터를 증강하는 생성기다.
골든 시드 1건을 입력받아, 원본 판정을 유지하는 변형 케이스 N건을 JSON 배열로만 출력한다.

[대전제]
- 이 태스크는 pep_excerpt(계획)와 rpt_excerpt(결과보고)를 대조해 criteria별로 충족/불가/검토를 판정한다.
- refs, 법조문, disclaimer, note 같은 추가 필드는 만들지 마라.
- input.criteria는 원본과 100% 동일하게 유지한다.
- output.label은 원본과 100% 동일하게 유지한다.
- output.eval의 criteria별 판정도 원본과 100% 동일하게 유지한다.
- 판정을 바꾸는 변형은 금지한다.
- 전체 label 계산 규칙은 다음과 같다.
  - criteria별 판정 중 불가가 1개 이상이면 output.label은 "불가"
  - 불가가 없고 검토가 1개 이상이면 output.label은 "검토"
  - 전부 충족이면 output.label은 "충족"

[eval 형식 규칙]
- output.eval은 반드시 문자열 배열이다.
- 각 원소는 반드시 다음 형식을 따른다.
  "특성: 판정 — 사유"
- 허용 특성명은 input.criteria에 포함된 것만 쓴다.
- 단, 정확성에는 "검토"를 쓰지 마라.

[criteria별 판정 로직 — 변형 시 이 경계를 반드시 보존]

1. 완전성(COMP)
- 충족:
  PEP에 열거된 과업, 범위, 산출물이 RPT에 모두 대응한다.
  동일 표현뿐 아니라 명백한 동의어와 패러프레이즈도 인정한다.
  같은 산출물, 같은 주체, 같은 시점을 가리키는 것이 명백하면 충족이다.
- 불가:
  1) PEP 항목 중 하나라도 RPT에 유사성조차 없이 없다.
  2) RPT가 해당 항목을 조건부 문구로만 처리한다.
     예: "필요 시", "~할 수 있다", "추후 검토"
- 검토:
  표현은 다르지만 같은 산출물, 같은 주체, 같은 시점인지 판단이 갈린다.
  즉, 대응 가능성은 있으나 동일 항목이라고 단정하기 어렵다.

2. 정확성(ACC)
- 충족:
  PEP의 계획값, 기준값, 기간, 예산, 건수, 규격과 RPT의 결과값이 직접 대조되어 일치한다.
- 불가:
  1) PEP 기준값과 RPT 결과값이 다르다.
  2) RPT 내부 수치나 서술에 자기모순이 있다.
  3) 비교 가능한 기준값 자체가 PEP와 RPT 발췌문에 없다.
- 정확성은 검토를 만들지 마라.

3. 검증가능성(VER)
- 블랙리스트 표현 예:
  "적절히", "최선을 다해", "원활히 협의하여", "필요한 범위에서", "우수한 수준으로", "성실히", "무리 없이", "신속히"
- 충족:
  결과 성과가 수치, 건수, 기준일, 비율, 시험결과, 검수결과, 측정값 등으로 확인 가능하게 서술된다.
  블랙리스트 표현이 있더라도 이를 제거한 뒤 남는 내용만으로 독립 검증 가능하면 충족으로 유지한다.
- 불가:
  1) 검증 서술 자체가 없다.
  2) 항목이 통째로 비어 있다.
  3) 블랙리스트 표현만 있고 독립 검증 가능한 기준이 없다.
- 검토:
  블랙리스트 표현이 있고, 이를 제거한 뒤에도 독립 검증 가능한지 판단이 갈리는 경우만 허용한다.

4. 추적성(TRACE)
- 충족:
  PEP의 단계, 과업, 산출물, 일정, 목표가 RPT 결과와 1:1로 명확히 대응된다.
  대응이 명확하면 추적성 판단 보조 신호 3개가 모두 없어도 충족으로 볼 수 있다.
- 불가:
  RPT의 결과 항목, 산출물, 단계가 PEP 어느 부분에도 대응하지 않거나, 반대로 PEP 핵심 단계가 RPT에서 대응 없이 끊긴다.
  이때 추적성 판단 보조 신호 ①산출물명 겹침 ②단계 순번·개수 대응 ③기능 설명 일치가 하나도 없어야 한다.
- 검토:
  단계명이나 산출물명이 달라졌거나 재구성되어 일부 대응 신호는 있으나 1:1 대응을 확정하기 어렵다.
  또는 현재 발췌문만으로 연결을 단정하기 어렵다.
  이때 추적성 판단 보조 신호 3개 중 일부만 있고 나머지는 없는 상태여야 한다.
  3신호가 모두 충족되어 1:1 대응이 명확해지면 충족으로 본다.
- 추적성 판단 보조 신호:
  ① 산출물명 겹침
  ② 단계 순번·개수 대응
  ③ 기능 설명 일치
- 위 3신호는 보조 판단 기준이다.
  다만 증강 시에는 다음 경계를 유지한다.
  · 불가: 3신호가 모두 없음
  · 검토: 3신호 중 일부만 있음
  · 충족: 1:1 대응이 명확함

[판정별 보존 규칙 — 위반 시 해당 케이스 폐기]
- 충족:
  원본의 대응 관계, 수치 일치, 검증 가능 기준, 1:1 연결 상태를 그대로 유지한다.
- 불가(완전성):
  원본에서 누락된 PEP 항목은 변형본에서도 계속 누락 상태여야 한다.
  우회 표현, 암시, 조건부 보완으로 결함을 해소하면 안 된다.
- 불가(정확성):
  원본의 불가 유형을 유지한다.
  가능한 유형은 1) PEP값과 RPT값 불일치 2) RPT 내부 자기모순 3) 비교 기준값 부재뿐이다.
- 불가(검증가능성):
  검증 가능한 기준이 없거나 비어 있는 상태를 유지한다.
- 불가(추적성):
  실제 대응 단절 상태를 유지한다.
  추적성 판단 보조 신호인 ①산출물명 겹침 ②단계 순번·개수 대응 ③기능 설명 일치가 하나도 없는 상태를 유지한다.
  위 3신호 중 하나라도 새로 넣어 대응 가능성이 생기게 만들면 안 된다.
- 검토(완전성):
  동일 항목인지 판단이 갈리는 애매함을 유지한다.
  특히 같은 산출물, 같은 주체, 같은 시점 중 하나 이상이 어긋나 판단이 갈리는 상태를 유지한다.
  명백히 같거나 명백히 다르게 만들면 안 된다.
- 검토(검증가능성):
  블랙리스트가 있고, 제거 후에도 독립 검증 가능 여부가 애매한 구조를 유지한다.
- 검토(추적성):
  일부 대응 신호는 있으나 1:1 연결을 단정하기 어려운 상태를 유지한다.
  추적성 판단 보조 신호 ①산출물명 겹침 ②단계 순번·개수 대응 ③기능 설명 일치 중 일부만 있고 나머지는 없는 상태를 유지한다.
  3신호를 모두 채워 1:1 대응이 명확해지는 형태로 바꾸면 안 된다.

[허용 변형 방식 — 골라 조합]

A. 표현 변형
- 같은 사실, 수치, 판정은 유지하고 문장구조, 어휘, 순서만 바꾼다.
- 표를 줄글로, 줄글을 항목 나열로 바꿀 수 있다.

B. 시나리오 교체
- 교육청, 대학, 공공기관 SW 사업 맥락 안에서 도메인만 바꾼다.
- 예: 학사행정, 학습관리, 상담챗봇, 학습분석, 정보보안, CCTV 관제, 포털 고도화
- 판정을 만드는 구조는 유지해야 한다.

C. 무해한 노이즈 추가
- 운영관리, 교육훈련, 개인정보보호, 보안점검, 인수인계, 유지관리 등 정상 항목을 rpt_excerpt에 1~3개 추가할 수 있다.
- 단, 노이즈가 누락, 불일치, 모호성, 추적 단절을 해소하면 안 된다.

[문체 규칙]
- 교육청, 대학, 공공기관 SW 사업 문맥을 유지한다.
- 수치는 현실 범위로 작성한다.
- 결과보고서 문체는 과거형 성과 서술을 사용한다.
  예: "구축하였다", "수행하였다", "산출하였다", "검증하였다", "적용하였다"
- eval은 반드시 "특성: 판정 — 사유" 문자열 배열로 쓴다.
- eval 사유는 어느 항목이 대응, 누락, 불일치, 모순, 모호, 단절되었는지 사실로만 서술한다.
- 규범적 표현이나 법률 판단어는 쓰지 않는다.

[출력 형식 — JSON 배열만]
[
  {
    "id": "<원본id>-aug1",
    "input": {
      "criteria": [...원본과 동일...],
      "pep_excerpt": "...",
      "rpt_excerpt": "..."
    },
    "output": {
      "label": "<원본 output.label과 동일>",
      "eval": [
        "완전성: 충족 — ...",
        "정확성: 충족 — ..."
      ]
    }
  }
]

[최종 점검]
각 케이스를 출력하기 전에 반드시 스스로 확인하라.
- input.criteria가 원본과 동일한가
- output.label이 원본과 동일한가
- output.eval의 각 특성 판정이 원본과 동일한가
- output.eval이 문자열 배열 형식인가
- 정확성에 "검토"를 넣지 않았는가
- 새 문장이 결함을 해소하지 않았는가
- JSON 배열 외 텍스트를 출력하지 않았는가

[입력 골든 시드]
{여기에 골든 1건}

위 시드와 동일한 판정을 유지하는 변형 {N}건을 생성하라.
"""

In [ ]:
# 증강 호출 + 파서
def parse_json(t):
    t = re.sub(r"^```(json)?|```$", "", t.strip(), flags=re.M).strip()
    return json.loads(t)

def augment(rec):
    prompt = RPT_AUG_PROMPT.replace("{N}", str(N)) \
             .replace("{여기에 골든 1건}", json.dumps(rec, ensure_ascii=False))
    resp = client.chat.completions.create(
        model=MODEL, temperature=TEMP,
        messages=[{"role":"user","content":prompt}],
    )
    return parse_json(resp.choices[0].message.content)

시드 개수: 30


In [ ]:
VALID = {"충족","불가","검토"}
def qc(rec, orig):
    if not {"seed_id","id","input","output"}.issubset(rec.keys()): return False, "필드누락"
    inp, out = rec["input"], rec["output"]
    if not {"criteria","pep_excerpt","rpt_excerpt"}.issubset(inp.keys()): return False, "input누락"
    if out.get("label") not in VALID: return False, "label이상"
    if out["label"] != orig["output"]["label"]: return False, "label변경됨"
    if set(inp["criteria"]) != set(orig["input"]["criteria"]): return False, "criteria변경됨"
    ev = out.get("eval")
    if not isinstance(ev, list) or not ev: return False, "eval형식"
    crit = set(inp["criteria"])
    for line in ev:
        feat = line.split(":")[0].strip()
        if feat not in crit: return False, f"criteria밖특성:{feat}"
        head = line.split("—")[0]
        if feat == "정확성" and "검토" in head: return False, "정확성검토금지"
    return True, ""

In [ ]:
random.seed(42)
aug, dropped = [], []
for rec in tqdm(seed_train):
    try:
        variants = augment(rec)
    except Exception as e:
        dropped.append({"seed_id": rec["seed_id"], "err": str(e)}); continue
    for i, v in enumerate(variants):
        v["seed_id"] = f"{rec['seed_id']}-aug{i+1}"   # 고유 식별자
        v["id"] = rec["id"]                            # 섹션 코드 복사
        ok, why = qc(v, rec)
        if not ok:
            dropped.append({"seed_id": v.get("seed_id"), "reason": why}); continue
        aug.append(v)

train_full = seed_train + aug
json.dump(train_full, open('RPT_train.json','w'), ensure_ascii=False, indent=2)
print(f"train시드 {len(seed_train)} + 증강 {len(aug)} = {len(train_full)} / 탈락 {len(dropped)}")
print("label 분포:", dict(Counter(r['output']['label'] for r in train_full)))
if dropped: print("탈락 샘플:", dropped[:5])

조합 개수: 66
